In [1]:
import logging, warnings
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr, ConstantInputWarning
from tqdm import tqdm
from umap import UMAP
from dtaidistance import dtw
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib
import importlib

from src.utils import iter_dataset_dirs, load_json
from src.plot_style import apply_plot_style
from src.visualization import plot_mts_corr_density, plot_spi_space_individual, plot_pca, plot_umap, plot_tsne

apply_plot_style()
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
warnings.simplefilter("ignore", ConstantInputWarning)


/Users/wedi0306/Library/CloudStorage/OneDrive-Personal/Desktop/2025USYD/USYD/mts-spi-study-cluster/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### In Development: Constructing MTS by randomly rolling g_affine, g_sigmoid, g_bell, with varying beta/noise (alpha0)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Literal, Optional, Tuple, List, Dict

import numpy as np

FilterName = Literal["affine", "sigmoid", "bell"]


def g_affine(x: np.ndarray, beta: float, alpha: float) -> np.ndarray:
    return beta * x + alpha


def g_sigmoid(x: np.ndarray, beta: float, alpha: float) -> np.ndarray:
    # stable-ish sigmoid; fine for your AR(1) scale
    return 1.0 / (1.0 + np.exp(-beta * x)) + alpha


def g_bell(x: np.ndarray, beta: float, alpha: float) -> np.ndarray:
    return np.exp(-beta * x * x) + alpha


_G_FUNCS = {
    "affine": g_affine,
    "sigmoid": g_sigmoid,
    "bell": g_bell,
}


@dataclass(frozen=True)
class ChannelSpec:
    channel: int
    g_name: FilterName
    beta: float
    alpha: float


@dataclass(frozen=True)
class MTSSampleMeta:
    # latent AR(1)
    T: int
    a: float
    noise_std: float
    z_seed: int

    # per-channel sampling
    M: int
    channel_seed: int  # controls g choice + beta + alpha draws

    # labeling helpers
    g_names: Tuple[FilterName, ...]            # length M
    counts: Dict[FilterName, int]              # counts per g
    positions: Dict[FilterName, Tuple[int, ...]]  # positions per g
    label: str                                 # compact deterministic label


def _make_label(g_names: Tuple[FilterName, ...]) -> str:
    # Example: "affine@0,3,7|sigmoid@1,4|bell@2,5,6,8,9,10,11"
    positions: Dict[FilterName, List[int]] = {"affine": [], "sigmoid": [], "bell": []}
    for i, g in enumerate(g_names):
        positions[g].append(i)
    parts = []
    for g in ("affine", "sigmoid", "bell"):
        idxs = ",".join(map(str, positions[g])) if positions[g] else "-"
        parts.append(f"{g}@{idxs}")
    return "|".join(parts)


def generate_mts_sample(
    M: int = 12,
    T: int = 1000,
    a: float = 0.8,
    noise_std: float = 1.0,
    z_seed: int = 0,
    channel_seed: int = 0,
    beta_mu: float = 1.0,
    beta_sigma: float = 0.25,
    alpha_low: float = 0.1,
    alpha_high: float = 1.0,
) -> Tuple[np.ndarray, MTSSampleMeta, Tuple[ChannelSpec, ...]]:
    """
    Returns:
      X: (M, T) observed MTS
      meta: dataset-level metadata + label
      specs: per-channel spec (g, beta, alpha) so you can reconstruct later
    """
    z = generate_ar1(T=T, a=a, noise_std=noise_std, seed=z_seed)

    rng = np.random.default_rng(channel_seed)

    g_names: List[FilterName] = []
    specs: List[ChannelSpec] = []
    X = np.empty((M, T), dtype=float)

    g_choices: np.ndarray = rng.choice(np.array(["affine", "sigmoid", "bell"], dtype=object), size=M)

    for i in range(M):
        g_name = str(g_choices[i])  # for mypy/typing
        beta = float(rng.normal(beta_mu, beta_sigma))
        alpha = float(rng.uniform(alpha_low, alpha_high))
        x_i = _G_FUNCS[g_name](z, beta=beta, alpha=alpha)

        X[i, :] = x_i
        g_names.append(g_name)  # type: ignore[arg-type]
        specs.append(ChannelSpec(channel=i, g_name=g_name, beta=beta, alpha=alpha))  # type: ignore[arg-type]

    g_names_t = tuple(g_names)

    counts = {k: 0 for k in ("affine", "sigmoid", "bell")}
    positions: Dict[FilterName, List[int]] = {"affine": [], "sigmoid": [], "bell": []}
    for i, g in enumerate(g_names_t):
        counts[g] += 1
        positions[g].append(i)

    positions_t = {k: tuple(v) for k, v in positions.items()}
    label = _make_label(g_names_t)

    meta = MTSSampleMeta(
        T=T,
        a=a,
        noise_std=noise_std,
        z_seed=z_seed,
        M=M,
        channel_seed=channel_seed,
        g_names=g_names_t,
        counts=counts,  # ok to keep as dict
        positions=positions_t,  # ok to keep as dict of tuples
        label=label,
    )
    return X, meta, tuple(specs)


def generate_mts_dataset(
    N: int,
    M: int = 12,
    T: int = 1000,
    a: float = 0.8,
    noise_std: float = 1.0,
    base_seed: int = 0,
) -> Tuple[np.ndarray, List[MTSSampleMeta], List[Tuple[ChannelSpec, ...]]]:
    """
    Generate many independent samples.

    Seeding strategy:
      - z_seed = base_seed + 2*k
      - channel_seed = base_seed + 2*k + 1
    so latent and channel draws differ but are deterministic.
    """
    Xs = np.empty((N, M, T), dtype=float)
    metas: List[MTSSampleMeta] = []
    specs_all: List[Tuple[ChannelSpec, ...]] = []

    for k in range(N):
        X, meta, specs = generate_mts_sample(
            M=M,
            T=T,
            a=a,
            noise_std=noise_std,
            z_seed=base_seed + 2 * k,
            channel_seed=base_seed + 2 * k + 1,
        )
        Xs[k] = X
        metas.append(meta)
        specs_all.append(specs)

    return Xs, metas, specs_all

In [ ]:

if __name__ == "__main__":
    N = 5
    Xs, metas, specs = generate_mts_dataset(N=N, M=12, T=1000, base_seed=0)

    print("Xs shape:", Xs.shape)  # (N, M, T)
    print("Sample 0 label:", metas[0].label)
    print("Sample 0 counts:", metas[0].counts)
    print("Sample 0 channel 0 spec:", asdict(specs[0][0]))


Xs shape: (5, 12, 1000)
Sample 0 label: affine@4,5,8,9|sigmoid@0,1,11|bell@2,3,6,7,10
Sample 0 counts: {'affine': 4, 'sigmoid': 3, 'bell': 5}
Sample 0 channel 0 spec: {'channel': 0, 'g_name': 'sigmoid', 'beta': 0.8657616911599287, 'alpha': 0.46827922273224515}
